# Оценка sentence embeddings на русскоязычных NLI-парах

Тема НИР: «Оценка способности русскоязычных sentence embeddings различать семантическую близость и логическое противоречие в парах предложений».

## Цель эксперимента

Оценить, насколько cosine similarity между русскоязычными sentence embeddings позволяет различать семантически близкие, нейтральные и логически противоречивые пары предложений.

In [ ]:
# Для запуска в Google Colab замените <YOUR_GITHUB_USERNAME> на реальный GitHub username.
# Если репозиторий уже склонирован и зависимости установлены, эту ячейку можно пропустить.
# !git clone https://github.com/<YOUR_GITHUB_USERNAME>/nir-sentence-embeddings.git
# %cd nir-sentence-embeddings
# !pip install -r requirements.txt

In [ ]:
import sys
from pathlib import Path

repo_path = Path.cwd()
if str(repo_path) not in sys.path:
    sys.path.append(str(repo_path))

# Альтернативно в Colab после clone можно явно указать путь:
# sys.path.append("/content/nir-sentence-embeddings")

In [ ]:
import pandas as pd
from matplotlib import pyplot as plt

from src.data import add_text_stats, load_xnli_ru, sample_balanced
from src.evaluation import (
    aggregate_by_label,
    compute_auc_entailment_vs_contradiction,
    compute_delta_entailment_contradiction,
    get_high_similarity_contradictions,
)
from src.models import compute_sentence_transformer_similarity, compute_tfidf_similarity
from src.visualization import plot_similarity_by_label

In [ ]:
df = load_xnli_ru(split="validation")
df.head()

In [ ]:
sample_df = sample_balanced(df, n_per_class=300, random_state=42)
sample_df = add_text_stats(sample_df)
sample_df["label"].value_counts()

In [ ]:
sample_df["tfidf_similarity"] = compute_tfidf_similarity(sample_df)
sample_df[["premise", "hypothesis", "label", "tfidf_similarity"]].head()

In [ ]:
model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
sample_df["st_similarity"] = compute_sentence_transformer_similarity(
    sample_df,
    model_name=model_name,
    batch_size=32,
)
sample_df[["premise", "hypothesis", "label", "st_similarity"]].head()

In [ ]:
aggregate_by_label(sample_df, "tfidf_similarity")

In [ ]:
aggregate_by_label(sample_df, "st_similarity")

In [ ]:
for score_column in ["tfidf_similarity", "st_similarity"]:
    delta = compute_delta_entailment_contradiction(sample_df, score_column)
    auc = compute_auc_entailment_vs_contradiction(sample_df, score_column)
    print(f"{score_column}: delta={delta:.4f}, ROC-AUC={auc:.4f}")

In [ ]:
fig = plot_similarity_by_label(sample_df, "st_similarity")
plt.show()

In [ ]:
columns = ["premise", "hypothesis", "label", "st_similarity", "lexical_overlap"]
get_high_similarity_contradictions(sample_df, "st_similarity", top_n=10)[columns]

## Черновик интерпретации результатов

После запуска нужно сравнить распределения similarity для `entailment`, `neutral` и `contradiction`. Если contradiction-пары часто получают высокую similarity, это может означать, что embedding-модель хорошо улавливает тематическую и лексическую близость, но хуже отражает логическое противоречие, особенно при отрицании или замене ключевого признака на противоположный.

Особое внимание стоит уделить таблице contradiction-пар с высокой cosine similarity: такие примеры помогут описать типичные ошибки модели и ограничения cosine similarity как самостоятельного критерия логического отношения между предложениями.